# SQL Worksheet — Week1

Use the following tables from the Riva Data Platform:

- `rivadataplatform.dataproduct.dim_batch`
- `rivadataplatform.dataproduct.dim_class`
- `rivadataplatform.dataproduct.fact_attendance`
- `rivadataplatform.dataproduct.dim_student`
- `rivadataplatform.dataproduct.dim_date`

**Instructions**
- Write SQL for each question.
- Do not modify the source data.
- Use clear aliases where JOINs are involved.
- Unless a question specifically asks for a particular column, select only the columns needed to answer it.


## Tables / Relationships

Useful keys:
- `dim_student.student_key` ↔ `fact_attendance.student_key`
- `dim_class.class_key` ↔ `fact_attendance.class_key`
- `dim_batch.batch_key` ↔ `fact_attendance.batch_key`
- `dim_class.batch_id` ↔ `dim_batch.batch_id`

## Question 1 — Student Attendance Profile
Using `dim_student`, `fact_attendance`, `dim_class`, and `dim_batch`, return one row per student and batch with the student name, batch name, number of attendance records, and the number of missing phone numbers. Include students with no attendance records. Group by every selected non-aggregated column and order by attendance records descending.

In [ ]:
SELECT
    s.student_id,
    s.student_name,
    COALESCE(b.batch_name, 'No batch') AS batch_name,
    COUNT(f.attendance_id) AS attendance_records,
    SUM(CASE WHEN NULLIF(s.phone_no, '') IS NULL THEN 1 ELSE 0 END) AS missing_phone_rows
FROM rivadataplatform.dataproduct.dim_student AS s
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.student_key = s.student_key
LEFT JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
GROUP BY s.student_id, s.student_name, b.batch_name
ORDER BY attendance_records DESC, s.student_name;

## Question 2 — Missing Profile Data by City
Join `dim_student` to attendance and class data. Group students by city and class topic, replace null city/topic values with readable labels, and return the distinct number of students, total attendance records, and missing phone count. Keep groups with at least one missing phone number.

In [ ]:
SELECT
    COALESCE(s.city, 'Unknown city') AS city,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    COUNT(DISTINCT s.student_key) AS distinct_students,
    COUNT(f.attendance_id) AS attendance_records,
    SUM(CASE WHEN NULLIF(s.phone_no, '') IS NULL THEN 1 ELSE 0 END) AS missing_phone_count
FROM rivadataplatform.dataproduct.dim_student AS s
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.student_key = s.student_key
LEFT JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
GROUP BY COALESCE(s.city, 'Unknown city'), COALESCE(c.topic, 'Topic not assigned')
HAVING SUM(CASE WHEN NULLIF(s.phone_no, '') IS NULL THEN 1 ELSE 0 END) > 0
ORDER BY missing_phone_count DESC, city, topic;

## Question 3 — Distinct Class Calendar
Join `fact_attendance` to `dim_class`, `dim_batch`, and `dim_date`. Return one row per distinct class date with the batch name, class topic (or `Topic not assigned` when null), calendar day name, and number of attendance records. Sort chronologically.

In [ ]:
SELECT DISTINCT
    c.class_date,
    b.batch_name,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    d.day_name,
    COUNT(f.attendance_id) OVER (PARTITION BY c.class_key) AS attendance_records
FROM rivadataplatform.dataproduct.dim_class AS c
JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.class_key = c.class_key
JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_id = c.batch_id
LEFT JOIN rivadataplatform.dataproduct.dim_date AS d
    ON d.date_key = f.date_key
ORDER BY c.class_date;

## Question 4 — Student and Class Detail
Join all five tables and return attendance detail for each student: student name, city, batch name, class date, day name, topic, and status. Replace null dimension values with labels, filter to records with a non-null attendance status, and sort by date then student name.

In [ ]:
SELECT
    s.student_name,
    COALESCE(s.city, 'Unknown city') AS city,
    COALESCE(b.batch_name, 'No batch') AS batch_name,
    c.class_date,
    COALESCE(d.day_name, c.class_day) AS day_name,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    f.attendance_status
FROM rivadataplatform.dataproduct.fact_attendance AS f
JOIN rivadataplatform.dataproduct.dim_student AS s
    ON s.student_key = f.student_key
JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_date AS d
    ON d.date_key = f.date_key
WHERE NULLIF(f.attendance_status, '') IS NOT NULL
ORDER BY c.class_date, s.student_name;

## Question 5 — Attendance Status Summary
Across each batch and class topic, calculate total attendance records, distinct students, and counts of Present, Late, and Absent records. Use null-safe topic and status labels, and return only groups containing at least one attendance record.

In [ ]:
SELECT
    COALESCE(b.batch_name, 'No batch') AS batch_name,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    COUNT(f.attendance_id) AS attendance_records,
    COUNT(DISTINCT f.student_key) AS distinct_students,
    SUM(CASE WHEN f.attendance_status = 'Present' THEN 1 ELSE 0 END) AS present_count,
    SUM(CASE WHEN f.attendance_status = 'Late' THEN 1 ELSE 0 END) AS late_count,
    SUM(CASE WHEN f.attendance_status = 'Absent' THEN 1 ELSE 0 END) AS absent_count
FROM rivadataplatform.dataproduct.fact_attendance AS f
LEFT JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
GROUP BY COALESCE(b.batch_name, 'No batch'), COALESCE(c.topic, 'Topic not assigned')
HAVING COUNT(f.attendance_id) > 0
ORDER BY batch_name, topic;

## Question 6 — Students Requiring Follow-up
Join students, attendance, classes, and batches to find students with at least one Late or Absent record. Return each student once with their batch, total attendance records, issue count, and most recent class date. Exclude students with no issue and order by issue count descending.

In [ ]:
SELECT
    s.student_id,
    s.student_name,
    COALESCE(MAX(b.batch_name), 'No batch') AS batch_name,
    COUNT(f.attendance_id) AS total_attendance_records,
    SUM(CASE WHEN f.attendance_status IN ('Late', 'Absent') THEN 1 ELSE 0 END) AS issue_count,
    MAX(c.class_date) AS most_recent_class_date
FROM rivadataplatform.dataproduct.dim_student AS s
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.student_key = s.student_key
LEFT JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
GROUP BY s.student_id, s.student_name
HAVING SUM(CASE WHEN f.attendance_status IN ('Late', 'Absent') THEN 1 ELSE 0 END) > 0
ORDER BY issue_count DESC, s.student_name;